# Unconstrained inverse-folding design across models

Runs **unconstrained** (no fixed positions) design on the glyco-benchmark protein set for each inverse-folding model and writes per-model FASTA designs + sequon-retention CSVs.

**Adding a new model**: drop a new adapter at `src/glyco_design/models/<model>.py` that subclasses `DesignModel`, then add a section below. Nothing else should need to change.

**Runtime**: each section assumes a GPU runtime. ProteinMPNN runs fastest (~1–2 min/protein), ESM-IF and TriFlow are slower.

> This notebook can run from a local SugarFix checkout or in Google Colab. In Colab, select **Runtime → Change runtime type → GPU** before running.

## Setup (run once)

Finds the local SugarFix checkout when available; otherwise clones it in Colab. Installs `glyco_design` so every model section can import adapters and the pipeline runner.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/LBDillon/SugarFix.git'

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    marker = Path('analysis/glycosylation-bias-analysis/src/glyco_design')
    for path in (start, *start.parents):
        if (path / marker).exists():
            return path
    return None

REPO_DIR = find_repo_root()
if REPO_DIR is None:
    REPO_DIR = Path('/content/SugarFix')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

ANALYSIS_DIR = REPO_DIR / 'analysis/glycosylation-bias-analysis'
SRC_DIR = ANALYSIS_DIR / 'src'

r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(ANALYSIS_DIR)],
    capture_output=True, text=True,
)
if r.returncode != 0:
    print(r.stdout); print(r.stderr)
    raise SystemExit('pip install -e failed')

# Editable installs write a .pth file that Python only reads at startup,
# so add src/ to sys.path so the running kernel sees the package immediately.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import glyco_design
print('repo:', REPO_DIR)
print('glyco_design ready:', glyco_design.__file__)


In [ ]:
# Shared config for every model section.
import os, sys
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

MANIFEST = ANALYSIS_DIR / 'data/glyco_benchmark/manifests/benchmark_manifest_simple.csv'
PDB_ROOT = ANALYSIS_DIR / 'data/glyco_benchmark/raw'
DRIVE_DESIGN_ROOT = Path('/content/drive/MyDrive/SugarFix/glyco_benchmark/designs')
DEFAULT_DESIGN_ROOT = DRIVE_DESIGN_ROOT if DRIVE_DESIGN_ROOT.parent.exists() else ANALYSIS_DIR / 'data/glyco_benchmark/designs'
DESIGN_ROOT = Path(os.environ.get('GLYCO_DESIGN_ROOT', DEFAULT_DESIGN_ROOT))

NUM_SEQS = int(os.environ.get('GLYCO_NUM_SEQS', 32))
TRIFLOW_NUM_SEQS = int(os.environ.get('GLYCO_TRIFLOW_NUM_SEQS', NUM_SEQS))
TEMPERATURE = float(os.environ.get('GLYCO_TEMPERATURE', 0.1))
PROTEIN_CLASS = 'glycoprotein'  # set to None to include controls
LIMIT = None  # set to a small int for smoke tests
CACHE_EXISTING = True
OVERWRITE_EXISTING = False

# Add folders of FASTA files you downloaded from earlier runs here.
# The pipeline looks for files named <PDB>_<CHAIN>_unconstrained.fasta.
EXISTING_FASTA_DIRS = {
    'proteinmpnn': [],
    'esm_if': [],
    'triflow': [],
}

os.makedirs(DESIGN_ROOT, exist_ok=True)
print('manifest:', MANIFEST)
print('pdb_root:', PDB_ROOT)
print('designs :', DESIGN_ROOT)
print('num_seqs:', NUM_SEQS, '| triflow_num_seqs:', TRIFLOW_NUM_SEQS, '| limit:', LIMIT)

## Optional cache check

Use this after filling `EXISTING_FASTA_DIRS` to see what downloaded FASTA files can be reused before rerunning a model.

In [ ]:
import pandas as pd
from glyco_design.pipeline import summarize_unconstrained_cache

cache_frames = []
for model_name, dirs in EXISTING_FASTA_DIRS.items():
    dirs = [Path(d) for d in dirs]
    if not dirs:
        continue
    cache_df = summarize_unconstrained_cache(
        manifest_path=MANIFEST,
        pdb_root=PDB_ROOT,
        fasta_dirs=dirs,
        protein_class=PROTEIN_CLASS,
        limit=LIMIT,
        num_seqs=TRIFLOW_NUM_SEQS if model_name == 'triflow' else NUM_SEQS,
    )
    cache_df.insert(0, 'model', model_name)
    cache_frames.append(cache_df)

cache_summary = pd.concat(cache_frames, ignore_index=True) if cache_frames else pd.DataFrame()
if len(cache_summary):
    display(cache_summary)
    display(cache_summary.groupby('model')['has_fasta'].agg(['sum', 'count']))
else:
    print('No extra FASTA cache directories configured yet.')

## ProteinMPNN

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec('colabdesign') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'git+https://github.com/sokrypton/ColabDesign.git@v1.1.0',
                    '--quiet'], check=True)
else:
    print('ColabDesign already installed')

In [ ]:
from glyco_design.models.proteinmpnn import ProteinMPNNDesignModel
from glyco_design.pipeline import run_unconstrained_experiment

model = ProteinMPNNDesignModel()
model.load()

df_mpnn = run_unconstrained_experiment(
    model=model,
    manifest_path=MANIFEST,
    pdb_root=PDB_ROOT,
    output_dir=f'{DESIGN_ROOT}/proteinmpnn',
    num_seqs=NUM_SEQS,
    temperature=TEMPERATURE,
    protein_class=PROTEIN_CLASS,
    limit=LIMIT,
    cache_existing=CACHE_EXISTING,
    overwrite=OVERWRITE_EXISTING,
    existing_fasta_dirs=EXISTING_FASTA_DIRS.get('proteinmpnn', []),
)
df_mpnn.head()

## ESM-IF

In [ ]:
# ESM-IF deps: facebookresearch/esm + Biotite + the PyG pieces used by ESM inverse folding.
import importlib.util, subprocess, sys, torch

TORCH_VER = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
PYG_WHL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA}.html'
print('torch:', torch.__version__, '| PyG wheels:', PYG_WHL)

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout); print(r.stderr)
        raise SystemExit(f'pip install {args} failed')

def ensure(module_name, *packages):
    if importlib.util.find_spec(module_name) is None:
        pip(*packages)
    else:
        print(module_name, 'already installed')

ensure('esm', 'fair-esm')
ensure('biotite', 'biotite')
ensure('torch_geometric', 'torch-geometric')

if importlib.util.find_spec('torch_scatter') is None:
    scatter_args = ['torch-scatter']
    if torch.version.cuda:
        scatter_args += ['-f', PYG_WHL]
    pip(*scatter_args)
else:
    print('torch_scatter already installed')

print('ESM-IF deps installed')


In [ ]:
from glyco_design.models.esm_if import ESMIFDesignModel
from glyco_design.pipeline import run_unconstrained_experiment

model = ESMIFDesignModel()
model.load()

df_esmif = run_unconstrained_experiment(
    model=model,
    manifest_path=MANIFEST,
    pdb_root=PDB_ROOT,
    output_dir=f'{DESIGN_ROOT}/esm_if',
    num_seqs=NUM_SEQS,
    temperature=TEMPERATURE,
    protein_class=PROTEIN_CLASS,
    limit=LIMIT,
    cache_existing=CACHE_EXISTING,
    overwrite=OVERWRITE_EXISTING,
    existing_fasta_dirs=EXISTING_FASTA_DIRS.get('esm_if', []),
)
df_esmif.head()

## TriFlow

Clones the TriFlow repo and loads the AFDB weights. TriFlow's loader expects weights at a relative path, so the adapter `chdir`s into the repo during `load()`.

In [ ]:
import os, subprocess, sys

TRIFLOW_DIR = '/content/TriFlow'

if not os.path.exists(TRIFLOW_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/jzhoulab/TriFlow.git', TRIFLOW_DIR], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'biopython==1.85', 'pytorch-lightning==2.5.1.post0',
                'deepspeed==0.16.7', 'ml-collections', 'einops', 'dm-tree'], check=True)

assert os.path.exists(f'{TRIFLOW_DIR}/weights/afdb_dataset/afdb_weights.pt'), \
    'TriFlow weights missing — see the TriFlow README for download instructions.'

In [ ]:
from glyco_design.models.triflow import TriFlowDesignModel
from glyco_design.pipeline import run_unconstrained_experiment

model = TriFlowDesignModel(triflow_dir=TRIFLOW_DIR)
model.load()

df_triflow = run_unconstrained_experiment(
    model=model,
    manifest_path=MANIFEST,
    pdb_root=PDB_ROOT,
    output_dir=f'{DESIGN_ROOT}/triflow',
    num_seqs=TRIFLOW_NUM_SEQS,
    temperature=TEMPERATURE,
    protein_class=PROTEIN_CLASS,
    limit=LIMIT,
    cache_existing=CACHE_EXISTING,
    overwrite=OVERWRITE_EXISTING,
    existing_fasta_dirs=EXISTING_FASTA_DIRS.get('triflow', []),
)
df_triflow.head()

## Caliby *(TODO)*

Caliby's sampling API is `model.sample(pdb_paths, num_seqs_per_pdb, temperature, pos_constraint_df=...)` — structurally simple to wrap. Adapter at `src/glyco_design/models/caliby.py` is the next step.

## Cross-model comparison

Concatenate per-model retention CSVs for a first-look comparison. Downstream analysis (experiments/04–09) reads from these.

In [ ]:
import pandas as pd

frames = []
for name, df in [('proteinmpnn', globals().get('df_mpnn')), ('esm_if', globals().get('df_esmif')), ('triflow', globals().get('df_triflow'))]:
    if df is not None and len(df):
        frames.append(df)

combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
combined.to_csv(f'{DESIGN_ROOT}/sequon_retention_unconstrained_all_models.csv', index=False)

if len(combined):
    summary = (
        combined.assign(preserved=combined['sequon_status'].isin(['NXS','NXT']))
                .groupby('model')['preserved'].mean()
                .rename('sequon_retention_rate')
                .to_frame()
    )
    summary['n_sites'] = combined.groupby('model').size()
    display(summary)